<div style="text-align: center;" >
<h1 style="margin-top: 0.2em; margin-bottom: 0.1em;">Relational Databases</h1>
<h4 style="margin-top: 0.7em; margin-bottom: 0.3em; font-style:italic"></h4>
</div>
<br>




### __1. Databases introduction__

A database is any logically modeled collection of information. This sounds overly simplified, but a database can be a lot of things and it means, it does not necessarily have to be stored on a computer.

An excel sheet, a stack of patient files in a hospital or a list of names in a text file could also be considered a database. 

But usually, in the context of computer science, it is more often thought as a set of structured tables that can be accessed via a *database management system* (DBMS). 

Could you give me examples where you think or know databases are used?

#### __1.1 Data Storage: Demands__

**Space Efficiency** 

In computational sciences datasets can be massive, which is why it is crucial to ensure space efficiency. Redundant information (storing the same data multiple times) can lead to unnecessary consumption of storage resources. To achieve space efficiency it is helpful to organize data into tables and linking the related information through relationships. This minimizes storage requirements and ensures that each piece of data is stored in a single place. 

**Consistency**  

This refers to the accuracy and reliability of data. It ensures that the information retrieved from a database is always valid and up-to-date. This is achieved by:
- Transaction Management: implement transactions to gorup multitple database operations into a single, atomic unit. This ensures that either all operations within a transaction are executed, or none of them are. 
- Concurrency Control: data storage should remain stable even when it is accessed by multitple users or processes concurrently. 

- Database Transaction-Properties: ACID

    - **A**tomicity : all or no transactions are committed
    - **C**onsistency: transaction completed or previous state is returned
    - **I**solation: transations are isolated from each other
    - **D**urability: completed transactions are saved securely

**Flexibility**


- Comfortable User Interface:
 Tools like SQL clients or custom applications should have intuitive layouts, making it easy for users to perform queries, updates, and other operations.


- All relevant information should be accessible

#### __1.2 Data Storage Solutions__


**Relational Databases:**
- representation through sets of linked tables that share a common key
- Suitable for structured data with well-defined relationships.
- SQL-based (Structured Query Language) systems like PostgreSQL, MySQL, or SQLite.

**NoSQL Databases:**
- allows different structures than a SQL database (not rows and columns) . 
    - Column-oriented
    - Key-value stores
    - Document stores
    - Graph databases
- more flexibility to use a format that best fits the data.
- NoSQL databases do sometimes support some SQL commands (sometimes defined as 'not only SQL')
- Examples include MongoDB, Cassandra, or CouchDB.





###  (bad) Example 

Assume, we want to store purchase information for some store:
- There is a set of Users with names, e-mail addresses, and IDs
- There is a set of Products with names, prices and IDs
- Users can make purchases which may consist of several products

In [1]:
import pandas as pd
import sqlite3

In [2]:
data = {
    'cust_ID': ['C1', 'C1', 'C1', 'C1', 'C2', 'C2'],
    'cust_name': ['Hans-Peter', 'Hans-Peter', 'Hans-Peter', 'Hans-Peter', 'Lea', 'Lea'],
    'e_mail': ['hp@uni.de', 'hp@uni.de', 'hp@uni.de', 'hp@uni.de', 'lea@uni.de', 'lea@uni.de'],
    'purch_id': ['C1_1', 'C1_1', 'C1_2', 'C1_2', 'C2_1', 'C2_1'],
    'purch_date': ['01-01-2020', '01-01-2020', '07-01-2020', '07-01-2020', '01-01-2020', '01-01-2020'],
    'prod_ID': ['P1', 'P2', 'P3', 'P4', 'P1', 'P4'],
    'prod_name': ['Toothpicks', 'Shampoo', 'Razor', 'Comb', 'Toothpicks', 'Comb'],
    'price': [1.99, 3.99, 9.99, 5.99, 1.99, 5.99]
}

df = pd.DataFrame(data)

df


,cust_ID,cust_name,e_mail,purch_id,purch_date,prod_ID,prod_name,price
0,C1,Hans-Peter,hp@uni.de,C1_1,01-01-2020,P1,Toothpicks,1.99
1,C1,Hans-Peter,hp@uni.de,C1_1,01-01-2020,P2,Shampoo,3.99
2,C1,Hans-Peter,hp@uni.de,C1_2,07-01-2020,P3,Razor,9.99
3,C1,Hans-Peter,hp@uni.de,C1_2,07-01-2020,P4,Comb,5.99
4,C2,Lea,lea@uni.de,C2_1,01-01-2020,P1,Toothpicks,1.99
5,C2,Lea,lea@uni.de,C2_1,01-01-2020,P4,Comb,5.99


What are possible problems with this structure?

#### __1.3 Anomalies__

- Insertion Anomaly
    - Adding information is artificially restricted by unrelated missing information
    - E.g. We can only add a Product to our Table if it has been purchased at least once
- Deletion Anomaly
    - Deletion of some Data may unintentionally destroy other information
    - E.g. Assume Hans-Peter cancels his second order and it gets deleted, all information on product P3 (Razor) is gone as well
- Modifying Anomaly
    - Data Structure is not robust against incorrect/partial changes
    - E.g. If we want to raise the price of toothpicks, we have to assure that both appearances get changed

What could we change to get a better structure?



#### One Solution:
- Split “Main”-Table into Subtables, s.t.
    - For each Subtable there exists a set of Variables which 
    determine all other Variables
        - We call this Set the primary Key of the Table
    - All Variables within a table are only dependent on the Key
    - For convenience we may add an artificial key variable (index)
- If needed, reassemble Information based on by joining Subtables based on their Key relationships
    - Tables may contain the primary Key of other Tables as foreign Key to express their relation


We can construct the following Table Structure: 

- customers: ( <u>cust_id</u>, cust_name, e_mail)
- purchases: (<u>purch_id</u>, cust_id → customer.cust_id, purch_date)
- purchase_contents: (<u>purch_id</u> → purchase.purch_id, <u>prod_id</u> → product.prid_id, quantity) #note: add quantitty to enable customers to buy more than one of a certain product
- products: (<u>prod_id</u>, prod_name, price)
    
• Keys are underlined and foreign keys are implied by pointing to their primary counterpart

**Note**:

The schema of a relational database consists of

- names of relations
- attributes (name, domain) of each relation
- contraints (primary and foreign keys)



<div class="alert alert-block alert-info">
    <b>Exercise</b>: Implement the Schema with SQLite
    
Note that if your working in VS Code you can download the 'SQLite Viewer' to have a easy look into your database.
</div>

In [3]:
# Implement the schema:
# Connect to a database (creates a new database if it doesn't exist)
conn = sqlite3.connect('Shopping.db')

# Create a cursor
cursor = conn.cursor()

# customers
cursor.execute("""CREATE TABLE IF NOT EXISTS customers (
  cust_id TEXT PRIMARY KEY,
  cust_name TEXT,
  e_mail TEXT
);
""")

# purchases
cursor.execute("""CREATE TABLE IF NOT EXISTS purchases (
  purch_id TEXT PRIMARY KEY,
  cust_id TEXT,
  purch_date DATETIME,
  FOREIGN KEY (cust_id) REFERENCES customers(cust_id)
);
""")

# products
cursor.execute("""CREATE TABLE IF NOT EXISTS products (
  prod_id TEXT PRIMARY KEY,
  prod_name TEXT,
  price REAL
);
""")

# products in each purchase
cursor.execute("""CREATE TABLE IF NOT EXISTS purchase_contents (
  purch_id TEXT,
  prod_id TEXT,
  quantity INT,
  PRIMARY KEY(purch_id, prod_id),
  FOREIGN KEY(purch_id) REFERENCES purchases(purch_id),
  FOREIGN KEY(prod_id) REFERENCES products(prod_id)
);""")

# Commit the changes
conn.commit()

# Close the connection
conn.close()

In [4]:
# Print all tables in the database:

conn = sqlite3.connect('Shopping.db')
cursor = conn.cursor()


# Query to fetch all table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

# Print table names
print("Tables in the database:")
for table in tables:
    print(table[0])  # Extract table name from the tuple

# Close the connection
conn.close()

Tables in the database:
customers
purchases
products
purchase_contents


<div class="alert alert-block alert-info">
    <b>Exercise</b>: Insert the data from your pandas dataframe, df, into your database.
</div>

In [5]:
customers_df = df[['cust_ID', 'cust_name', 'e_mail']].drop_duplicates().reset_index(drop=True)
customers_df

,cust_ID,cust_name,e_mail
0,C1,Hans-Peter,hp@uni.de
1,C2,Lea,lea@uni.de


In [6]:
# Hint if you run these insert code chunks twice on the same database you will get a keys-are-not-unique error.
# checkout the if_exits argument in the to_sql function or if you want to start the database from scratch, you can just delete the .db file containing the database.

conn = sqlite3.connect('Shopping.db')

# instead of if_exists='append' also possible to choose 'replace' - deletes everything currently in the table and adds the new values
customers_df.to_sql('customers', conn, if_exists='append', index=False) 

conn.close()


In [7]:
purchases_df = df[['purch_id', 'cust_ID', 'purch_date']].drop_duplicates().reset_index(drop=True)
purchases_df

,purch_id,cust_ID,purch_date
0,C1_1,C1,01-01-2020
1,C1_2,C1,07-01-2020
2,C2_1,C2,01-01-2020


In [8]:
products_df = df[['prod_ID', 'prod_name', 'price']].drop_duplicates().reset_index(drop=True)
products_df

,prod_ID,prod_name,price
0,P1,Toothpicks,1.99
1,P2,Shampoo,3.99
2,P3,Razor,9.99
3,P4,Comb,5.99


In [9]:
purchase_contents_df = df[['purch_id', 'prod_ID']].drop_duplicates().reset_index(drop=True)
purchase_contents_df['quantity'] = 1 # add the quantities column
purchase_contents_df

,purch_id,prod_ID,quantity
0,C1_1,P1,1
1,C1_1,P2,1
2,C1_2,P3,1
3,C1_2,P4,1
4,C2_1,P1,1
5,C2_1,P4,1


In [10]:
# you can also run several to_sql commands in one chunk

conn = sqlite3.connect('Shopping.db')

purchases_df.to_sql('purchases', conn, if_exists='append', index=False)
products_df.to_sql('products', conn, if_exists='append', index=False)
purchase_contents_df.to_sql('purchase_contents', conn, if_exists='append', index=False)

conn.close()


<div class="alert alert-block alert-info">
    <b>Exercise</b>: Assume a University with lecturers, students, courses and faculties. 
    
Find a Database scheme that expresses the following information:
- Each lecturer is an employee of a single faculty
- Students are enrolled in Faculties
- Lecturers give courses which belong to a faculty (not necessarily to the lecturer’s faculty). Only one teacher per course
- Students visit Courses and score Grades in them
- Lecturers and Students have a unique ID and a name
- Courses have a unique ID and a name, they give a fixed amount 
of credits
- Additionally, we want to store the number of semesters a 
students took so far and the lecturers’ salaries.

</div>

#### One Solution


Faculties(<u>f_id</u>, name )

Lecturers(<u>l_id</u>, name, faculty → faculties.f_id, salary)

Students(<u>s_id</u>, name, faculty → faculties.f_id, semester)

takes(<u>course</u> → courses.c_id, <u>student</u> → students.s_id, grade)

Courses(<u>c_id</u>, name, credits, faculty → faculties.f_id, lecturer → lecturers.l_id)

### __2. SQL__

Declarative Database Language

consists of three (co-dependent) language definitions
- data definition language (DDL)
    - define relational schemata
    - create/ alter/ delete tables and their attributes
- data manipulation language (DML)
    - insert/ delete/ modify entries in tables
    - query one or more tables
- data control language (DCL)
    - control access to data stored in database
    - grant/ revoke/ modify user access rights

#### __2.1 DDL__

The basic task of a **D**ata **D**efinition **L**anguage is to be able to declare the elements of a database schema, i.e., to communicate the result of the database design process to the database system. 

**CREATE TABLE**

    CREATE TABLE table_name AS (
    column1 datatype,
    column2 datatype,
    column2 datatype,
    PRIMARY KEY(column1) 
    )

- Large List of possible Datatypes: e.g. (var)char, integer, float
- Add foreign Keys by adding row

    `FOREIGN KEY(column1) REFERENCES other_table(key_attr)`

**ALTER TABLE**

    ALTER TABLE table_name ADD (column99 datatype)


**DROP TABLE**

    DROP TABLE table_name

#### __2.2 DML__

The **D**ata **M**anipulation **L**anguage is used to modify the state of a database, i.e., to add, modify in and remove from tables


**INSERT INTO**


    INSERT INTO table_name
    VALUES
    (val_1_1, val_1_2, … val_1_m),
    (val_1_1, val_1_2, … val_1_m),
    …
    (val_n_1, val_n_2, … val_n_m)

- The Inserted Values must fit the column datatypes of the table
- SQL automatically checks for violation of key or other constraint

**DELETE FROM**

    DELETE FROM table_name [WHERE condition]

- if the search condition is emitted, all tuples are deleted. 

**UPDATE**

    UPDATE table_name 
    set val_1_1 = new_val_1_1
    [where condition]

**Queries**

basic syntax:

    SELECT (DISTINCT) column1, column2, …
    FROM table_name
    WHERE Condition 


Further possibility:

    SELECT
    FROM            JOIN        ON 
    WHERE       
    (AND)
    GROUP BY
    HAVING
    ORDER BY


- We can merge multiple tables in FROM statement using JOIN-Operations
- We can also order return data
- We may even group data and compute some group attributes using GROUP BY
- Set Operations like INTERSECT and UNION are allowed as well

### __3. SQL in Python__

First, we implement the following schema: 

- Students(
  s_id,          -- PRIMARY KEY
  name
)
- Professors(
  p_id,          -- PRIMARY KEY
  name
)
- Courses(
  c_id,          -- PRIMARY KEY
  title,
  credits
)
- Teaches(
  professor → Professors.p_id,
  course    → Courses.c_id,
  semester,
  PRIMARY KEY (professor, course, semester)
)
- Takes_course(
  student → Students.s_id,
  course  → Courses.c_id,
  semester,
  grade,
  PRIMARY KEY (student, course, semester)
)

The main goal of this schema is to model the academic process over time, focusing on who teaches what and who takes what, in which semester.

**Important Note**: Execute the following cell only once (!) and then delete or comment out the create table and insert into statements!

In [11]:
# Connect to a database (creates a new database if it doesn't exist)
conn = sqlite3.connect('DB_tutorial.db')

# Create a cursor
cursor = conn.cursor()

# Query data
cursor.execute("""CREATE TABLE students (
  id INT PRIMARY KEY,
  name text
);
               """)

cursor.execute("""CREATE TABLE courses (
  code INT PRIMARY KEY,
  title text,
  credits INT
);
               """)
cursor.execute("""CREATE TABLE professors (
  id INT PRIMARY KEY,
  name text
);""")

cursor.execute("""CREATE TABLE teaches (
  professor_id INT,
  course_code INT,
  semester CHAR(7),
  PRIMARY KEY(professor_id, course_code, semester),
  FOREIGN KEY(professor_id)
  	REFERENCES professors(id),
  FOREIGN KEY(course_code)
  	REFERENCES courses(code)
);""")

cursor.execute("""CREATE TABLE takes_course (
  student_id INT,
  course_code INT,
  semester CHAR(7),
  grade FLOAT,
  PRIMARY KEY(student_id, course_code, semester),
  FOREIGN KEY(student_id)
  	REFERENCES students(id),
  FOREIGN KEY(course_code)
  	REFERENCES courses(code)
);""")


#Insert the data/information 
cursor.execute("""INSERT INTO students 
VALUES 
(1, 'Dora Nynke'),
(2, 'Stefan Anit'),
(3, 'Tone Callista'),
(4, 'Amira Imtiyaz'),
(5, 'Anna Hamed')
;""")
conn.commit() 

cursor.execute("""INSERT INTO courses 
VALUES 
(111, 'Biology', 6),
(222, 'ICSS', 9),
(333, 'Introduction to Finance', 8)
;""")
conn.commit() 
cursor.execute("""INSERT INTO professors 
VALUES 
(11, 'Brychan Ritu'),
(22, 'David Garcia'),
(33, 'Steph Victoriano'),
(44, 'Miroslava Bernarda')
;
""")
conn.commit() 

cursor.execute("""INSERT INTO teaches 
VALUES
(11, 111, 'WS22/23'),
(22, 222, 'WS22/23'),
(33, 333, 'WS22/23'),
(44, 333, 'WS22/23')
;""")
conn.commit() 

cursor.execute("""INSERT INTO takes_course 
VALUES
(1, 111, 'WS22/23', 1.7),
(2, 222, 'WS22/23', 2.3),
(2, 333, 'WS22/23', 1.3),
(4, 333, 'WS22/23', 1.0),
(5, 111, 'WS22/23', 3.0),
(5, 222, 'WS22/23', 1.3),
(5, 333, 'WS22/23', 2.0)
;""")
conn.commit() 


# check upload
cursor.execute('SELECT * FROM students')

rows = cursor.fetchall()
for row in rows:
    print(row)


# Close the connection
conn.close()

(1, 'Dora Nynke')
(2, 'Stefan Anit')
(3, 'Tone Callista')
(4, 'Amira Imtiyaz')
(5, 'Anna Hamed')


<div class="alert alert-block alert-info">
    <b>Exercise</b>: 
Write the following queries:

- Select the names from all the students
- Select all students' names and id that take ICSS
- Select all students' names and id taught by David Garcia
- Could the database be improved? How?

</div>

In [12]:
# your solution
conn = sqlite3.connect('DB_tutorial.db')

cursor = conn.cursor()

# check upload
cursor.execute('SELECT name FROM students')

rows = cursor.fetchall()
for row in rows:
    print(row)

# Close the connection
conn.close()

('Dora Nynke',)
('Stefan Anit',)
('Tone Callista',)
('Amira Imtiyaz',)
('Anna Hamed',)


In [13]:
# your solution
conn = sqlite3.connect('DB_tutorial.db')

cursor = conn.cursor()

# check upload
cursor.execute("""SELECT students.id, students.name
FROM students
INNER JOIN takes_course ON students.id = takes_course.student_id
INNER JOIN courses ON takes_course.course_code = courses.code
WHERE courses.title = 'ICSS';
""")

rows = cursor.fetchall()
for row in rows:
    print(row)

# Close the connection
conn.close()

(2, 'Stefan Anit')
(5, 'Anna Hamed')


In [14]:
conn = sqlite3.connect('DB_tutorial.db')
cursor = conn.cursor()
cursor.execute(""" SELECT students.id, students.name FROM students
               JOIN takes_course ON students.id = takes_course.student_id
               JOIN courses ON takes_course.course_code = courses.code
               JOIN teaches ON courses.code = teaches.course_code
               JOIN professors ON teaches.professor_id = professors.id
               WHERE professors.name = 'David Garcia';
""")

rows = cursor.fetchall()
for row in rows: 
    print(row)
conn.close()

(2, 'Stefan Anit')
(5, 'Anna Hamed')
